# 🛵 Laboratorio: DBSCAN con coordenadas GPS — el caso Entregas Monterrey

## 📖 La historia
Una empresa regiomontana de entregas de **última milla** surte todo el área metropolitana de Monterrey desde **un solo almacén**. Los repartidores pasan más tiempo en el periférico que entregando, y la directora de operaciones quiere abrir **micro-centros de distribución** (bodegas pequeñas, cerca del cliente) donde la demanda se concentra.

Tenemos las coordenadas GPS de **2,100 entregas** de un mes. Nadie nos dice cuántas zonas hay, y una parte de las entregas cae en lugares sueltos donde **no** vale la pena poner nada.

> ### ❓ ¿Dónde abrir micro-centros de distribución y qué zona cubriría cada uno?

¿Por qué DBSCAN y no K-Means? Porque no sabemos cuántas zonas hay, porque una zona de demanda **no tiene por qué ser redonda** y, sobre todo, porque K-Means **obligaría** a asignar cada entrega suelta a algún micro-centro.

## 🧭 Lo que vas a aprender
1. Por qué **no** se usa la distancia euclidiana sobre grados de latitud y longitud
2. Usar DBSCAN con **`metric='haversine'`**: coordenadas en **radianes** y `eps = km / 6371`
3. Elegir `eps` con una gráfica de k-distancia **en kilómetros**
4. Convertir clusters en decisiones: **centroide**, **tamaño** y **radio** de cada zona
5. Ver qué pasa cuando K-Means tiene que asignar **el ruido**

## ⏱️ Agenda (segunda mitad de la sesión: 90 min)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación | 5 min |
| 1 | Las entregas en el mapa | 10 min |
| 2 | ¿Por qué no la euclidiana sobre grados? | 15 min |
| 3 | Haversine y k-distancia en kilómetros | 15 min |
| 4 | DBSCAN: las zonas de alta concentración | 15 min |
| 5 | De clusters a micro-centros | 15 min |
| 6 | ¿Y si usamos K-Means? | 10 min |
| 7 | Conclusiones | 5 min |

## 📋 Las variables
| Variable | Descripción |
|---|---|
| `entrega_id` | Identificador de la entrega |
| `lat` | Latitud en **grados** (positiva = norte del ecuador; Monterrey ≈ 25.7°) |
| `lon` | Longitud en **grados** (negativa = oeste de Greenwich; Monterrey ≈ -100.3°) |
| `hora` | Hora del día en que se entregó (8 a 21) |
| `monto` | Valor del pedido en **pesos** |

> 💡 Los datos son **sintéticos** (simulados con semilla fija), pero las zonas están en lugares reales del área metropolitana.

---
# 0️⃣ Preparación (5 min)

Ejecuta las siguientes celdas **una sola vez**.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_silueta_k(tabla)` | Silhouette promedio según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta_k(tabla):
    """Silhouette promedio para cada k (tabla de tabla_metricas_k). Más alto es mejor."""
    fig = px.line(
        tabla,
        x='k',
        y='silhouette',
        markers=True,
        labels={'k': 'Número de clusters (k)', 'silhouette': 'Silhouette promedio'},
        title='Silhouette según k (↑ mejor)'
    )
    fig.update_xaxes(dtick=1)
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido y silhouette (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y silhouette en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [ ]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Silhouette de un agrupamiento. El ruido (-1) se excluye del cálculo,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
    else:
        fila['silhouette'] = np.nan
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con la inercia (para el codo) y la silhouette."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), 'silhouette': fila['silhouette']})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

### 📥 Los datos
La siguiente celda **simula** las entregas: varias zonas de alta concentración con distinta dispersión y entregas sueltas por toda la ciudad. Siempre genera los mismos datos (`seed=42`).

In [ ]:
# Datos del caso entregas
import numpy as np
import pandas as pd

# Zonas con muchas entregas: (nombre, latitud, longitud, dispersión en km, entregas, hora pico, ticket promedio en $)
ZONAS_MTY = [
    ('Centro / Macroplaza',       25.6697, -100.3097, 0.5, 450, 13, 350),
    ('San Pedro / Valle Oriente', 25.6450, -100.3500, 0.6, 400, 12, 650),
    ('San Nicolás',               25.7440, -100.2950, 1.0, 350, 18, 400),
    ('Cumbres',                   25.7350, -100.4050, 1.2, 300, 19, 450),
    ('Apodaca / aeropuerto',      25.7750, -100.1500, 1.8, 250, 10, 900),
    ('Santa Catarina',            25.6730, -100.4580, 0.9, 150, 11, 500),
]


def generar_entregas(seed=42, n_dispersas=200):
    """Simula un mes de entregas en el área metropolitana de Monterrey.
    Cada zona es una nube de puntos alrededor de su centro (dispersión en km);
    además hay entregas dispersas por toda la ciudad."""
    rng = np.random.default_rng(seed)
    partes = []
    for nombre, lat0, lon0, sd_km, n, hora_pico, ticket in ZONAS_MTY:
        norte_km = rng.normal(0, sd_km, n)
        este_km = rng.normal(0, sd_km, n)
        partes.append(pd.DataFrame({
            'lat': lat0 + norte_km / 111.0,                                # 1° de latitud ≈ 111 km
            'lon': lon0 + este_km / (111.32 * np.cos(np.radians(lat0))),   # 1° de longitud ≈ 100 km aquí
            'hora': rng.normal(hora_pico, 2.5, n),
            'monto': ticket * rng.lognormal(-0.125, 0.5, n),
        }))
    partes.append(pd.DataFrame({
        'lat': rng.uniform(25.60, 25.82, n_dispersas),
        'lon': rng.uniform(-100.50, -100.08, n_dispersas),
        'hora': rng.uniform(8, 21, n_dispersas),
        'monto': 450 * rng.lognormal(-0.125, 0.5, n_dispersas),
    }))
    df = pd.concat(partes, ignore_index=True)
    df['hora'] = df['hora'].round().clip(8, 21).astype(int)
    df['monto'] = df['monto'].round(2)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)   # revolvemos el orden
    df.insert(0, 'entrega_id', np.arange(1, len(df) + 1))
    return df

### 🧩 Funciones del caso

| Función | ¿Para qué sirve? |
|---|---|
| `plot_mapa(df, labels=None, centros=None)` | Entregas sobre un mapa real (OpenStreetMap), coloreadas por zona; el ruido en gris. `centros` agrega los micro-centros |
| `plot_k_distancia_km(X_rad, k, eps_km=None)` | Gráfica de k-distancia con Haversine, **en kilómetros** |
| `resumen_zonas(df, labels)` | Por zona: entregas, centroide (lat, lon), radio que cubre al 90% de sus entregas, monto total, ticket promedio y hora típica |

In [ ]:
# Funciones del caso
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics.pairwise import haversine_distances
from sklearn.neighbors import NearestNeighbors

R_TIERRA_KM = 6371.0      # radio medio de la Tierra


def plot_mapa(df, labels=None, centros=None, title='Entregas en el área metropolitana de Monterrey'):
    """Entregas sobre un mapa real. labels (opcional) colorea por zona (-1 = Ruido, en gris).
    centros (opcional): DataFrame con columnas lat y lon (p. ej. la salida de resumen_zonas)."""
    datos = df[['lat', 'lon', 'hora', 'monto']].copy()
    color = None
    if labels is not None:
        datos['zona'] = _etiquetas_texto(labels)
        color = 'zona'
    fig = px.scatter_map(
        datos,
        lat='lat',
        lon='lon',
        color=color,
        category_orders=_orden_clusters(datos['zona']) if labels is not None else None,
        color_discrete_map={'Ruido': 'lightgray'},
        hover_data=['hora', 'monto'],
        opacity=0.7,
        zoom=10.3,
        center={'lat': 25.71, 'lon': -100.30},
        map_style='open-street-map',
        title=title
    )
    if centros is not None:
        fig.add_trace(go.Scattermap(
            lat=centros['lat'],
            lon=centros['lon'],
            mode='markers+text',
            marker={'size': 16, 'color': 'black'},
            text=[str(i) for i in centros.index],
            textposition='top right',
            name='Micro-centro'
        ))
    fig.update_traces(marker={'size': 6}, selector={'type': 'scattermap', 'mode': 'markers'})
    fig.update_layout(width=900, height=650, margin={'l': 10, 'r': 10, 't': 50, 'b': 10})
    fig.show()


def plot_k_distancia_km(X_rad, k, eps_km=None):
    """Como plot_k_distancia, pero con distancia Haversine y en kilómetros. X_rad = [lat, lon] en radianes.
    Usa k = min_samples: cada entrega cuenta como una de sus k, igual que en DBSCAN."""
    # n_neighbors=k incluye a la propia entrega (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k, metric='haversine').fit(X_rad)
    d_k = np.sort(vecinos.kneighbors(X_rad)[0][:, -1]) * R_TIERRA_KM
    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Entregas ordenadas por distancia', 'y': f'Radio para tener {k} entregas, incluida ella (km)'},
        title=f'Gráfica de k-distancia en km (k = {k})'
    )
    if eps_km is not None:
        fig.add_hline(y=eps_km, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps_km} km', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def resumen_zonas(df, labels):
    """Una fila por zona (sin el ruido): entregas, centroide, radio que cubre al 90% de sus entregas,
    monto total, ticket promedio y hora típica (mediana)."""
    labels = np.asarray(labels)
    filas = []
    for z in sorted(set(labels) - {-1}):
        zona = df[labels == z]
        centro = [[zona['lat'].mean(), zona['lon'].mean()]]
        dist_km = haversine_distances(np.radians(zona[['lat', 'lon']]), np.radians(centro))[:, 0] * R_TIERRA_KM
        filas.append({
            'zona': z,
            'entregas': len(zona),
            'pct': round(100 * len(zona) / len(df), 1),
            'lat': round(centro[0][0], 4),
            'lon': round(centro[0][1], 4),
            'radio_90_km': round(np.percentile(dist_km, 90), 2),
            'monto_total': round(zona['monto'].sum()),
            'ticket_promedio': round(zona['monto'].mean()),
            'hora_tipica': int(zona['hora'].median()),
        })
    return pd.DataFrame(filas).set_index('zona')

---
# 1️⃣ Las entregas en el mapa (10 min)

In [ ]:
# 1. df = generar_entregas(seed=42)
# 2. Imprime df.shape y muestra df.describe().round(2)
# ✍️ Tu código aquí


In [ ]:
# Dibuja todas las entregas en el mapa
# Pista: plot_mapa(df)
# ✍️ Tu código aquí


> 💡 El mapa de fondo se descarga de OpenStreetMap al abrir el notebook. Si no tienes internet verás los puntos sobre fondo blanco: el análisis es el mismo.

✍️ **¿Qué observas?**
- ¿Entre qué valores se mueven `lat` y `lon`? ¿Y el monto y la hora?
- ¿Cuántas zonas de alta concentración ves a simple vista? ¿Todas son igual de compactas?

RELLENAR CON TUS COMENTARIOS

---
# 2️⃣ ¿Por qué no la euclidiana sobre grados? (15 min)

La tentación: `DBSCAN(eps=0.01).fit(df[['lat', 'lon']])`. Hay dos problemas.

**Problema 1: un grado no mide lo mismo en las dos direcciones.** La Tierra es (casi) una esfera:
- Un grado de **latitud** (norte-sur) mide siempre ≈ **111 km**.
- Un grado de **longitud** (oriente-poniente) se **encoge** conforme te alejas del ecuador: mide $111.32 \cdot \cos(\text{latitud})$ km. En el ecuador, 111 km; en los polos, 0.

**Problema 2: ¿cuánto es `eps = 0.01`?** Nadie en operaciones piensa en grados. Queremos poder decir "un radio de **X kilómetros**".

Veamos cuánto mide un grado aquí, en Monterrey:

In [ ]:
lat_mty = 25.7
km_por_grado_lat = 111.0
km_por_grado_lon = 111.32 * np.cos(np.radians(lat_mty))

print(f'1° de latitud  ≈ {km_por_grado_lat:.1f} km')
print(f'1° de longitud ≈ {km_por_grado_lon:.1f} km   (a {lat_mty}° N)')
print(f'1° de longitud en Oslo (60° N) ≈ {111.32 * np.cos(np.radians(60)):.1f} km')

Ahora con la fórmula de **Haversine** (distancia real sobre la esfera) comparamos dos pares de entregas separadas **exactamente 0.01°**: uno hacia el **norte** y otro hacia el **oriente**. Para la euclidiana sobre grados, ambos pares están a la misma distancia (0.01).

`haversine_distances` de scikit-learn recibe `[lat, lon]` en **radianes** y regresa un ángulo; multiplicado por el radio de la Tierra da kilómetros.

In [ ]:
macroplaza = [25.6697, -100.3097]
al_norte = [25.6797, -100.3097]       # +0.01° de latitud
al_oriente = [25.6697, -100.2997]     # +0.01° de longitud

# 1. d_norte = haversine_distances(np.radians([macroplaza]), np.radians([al_norte]))[0, 0] * R_TIERRA_KM
# 2. d_oriente = lo mismo, pero con al_oriente
# 3. Imprime ambas distancias y la diferencia en %: 100 * (d_norte / d_oriente - 1)
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Cuántos km mide un grado de longitud en Monterrey comparado con uno de latitud? ¿Y en Oslo?
- ¿Qué diferencia (en %) hay entre 0.01° al norte y 0.01° al oriente? ¿Qué pasaría con un radio `eps` en grados?

RELLENAR CON TUS COMENTARIOS

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# Proyección local a kilómetros (sólo la usaremos para K-Means y para las métricas)
lat_ref = df['lat'].mean()
X_km = np.column_stack([
    (df['lon'] - df['lon'].mean()) * 111.32 * np.cos(np.radians(lat_ref)),   # km hacia el oriente
    (df['lat'] - df['lat'].mean()) * 111.0                                    # km hacia el norte
])
X_km[:3].round(2)

---
# 3️⃣ Haversine y k-distancia en kilómetros (15 min)

### Paso 1: coordenadas en radianes
Con `metric='haversine'` scikit-learn exige:
- Las columnas en orden **`[lat, lon]`** (latitud primero).
- Los valores en **radianes**: `np.radians(...)`.
- El `eps` también en radianes: un radio de $d$ km sobre una esfera de radio $R$ es un ángulo de $d / R$. Por eso **`eps = km / 6371`**.

⚠️ **No escalamos** las coordenadas: latitud y longitud ya están en la misma unidad (ángulos) y Haversine las combina correctamente. Escalarlas destruiría la geometría.

In [ ]:
# Convierte las columnas ['lat', 'lon'] a radianes y guárdalas en X_rad
# Pista: np.radians(df[['lat', 'lon']].to_numpy())
# ✍️ Tu código aquí


### Paso 2: `min_samples`
La regla $2 \times$ dimensiones daría **4**: con 2,100 entregas, cualquier grupito de 4 casas vecinas sería una "zona". Para un micro-centro necesitamos **demanda**: pedimos **`min_samples = 20`** (≈ 1% de las entregas). Un punto núcleo es una entrega con al menos 20 entregas del mes (contándose a sí misma) en su radio.

> 💡 La regla de dimensiones es un **mínimo**. Con muchos datos y ruido, súbela; y si el parámetro tiene una lectura de negocio (aquí, "demanda mínima"), úsala.

### Paso 3: la gráfica de k-distancia en km (k = 20)

In [ ]:
# Gráfica de k-distancia en km con k = 20
# Pista: plot_k_distancia_km(X_rad, k=...)
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
vecinos_20 = NearestNeighbors(n_neighbors=20, metric='haversine').fit(X_rad)   # incluye a la propia entrega
distancia_20_km = vecinos_20.kneighbors(X_rad)[0][:, -1] * R_TIERRA_KM

pd.DataFrame({
    'eps_km': [0.5, 0.8, 1.0, 1.2, 1.5],
    'pct_no_nucleo': [round(100 * (distancia_20_km > e).mean(), 1) for e in [0.5, 0.8, 1.0, 1.2, 1.5]]
})

✍️ **¿Qué observas?**
- ¿Dónde está el codo de la curva (en km)?
- Según la tabla, ¿qué % de entregas no sería núcleo con cada `eps`?
- ¿Qué `eps` en km elegirías? ¿Cómo se lo explicarías a la directora de operaciones?

RELLENAR CON TUS COMENTARIOS

---
# 4️⃣ DBSCAN: las zonas de alta concentración (15 min)

Recuerda: el `eps` va en **radianes** → `eps = km / R_TIERRA_KM`.

In [ ]:
# 1. eps_km = <tu radio en km>
# 2. dbscan_entregas = DBSCAN(eps=eps_km / R_TIERRA_KM, min_samples=20, metric='haversine').fit(X_rad)
# 3. labels_zona = dbscan_entregas.labels_
# 4. comparacion = []  y  comparacion.append(evaluar_clustering('DBSCAN Haversine', X_km, labels_zona))
#    (evaluamos en X_km para que las métricas estén en km)
# 5. pd.DataFrame(comparacion)
# ✍️ Tu código aquí


In [ ]:
# Dibuja el mapa coloreado por zona
# Pista: plot_mapa(df, labels_zona)
# ✍️ Tu código aquí


### ¿Qué tan sensible es el resultado al radio?
Si un cambio pequeño en `eps` cambia todo, no podemos confiar en la recomendación. Probamos varios radios:

In [ ]:
# Para km en [0.5, 0.8, 1.0, 1.2, 1.5]:
#   1. labels_km = DBSCAN(eps=km / R_TIERRA_KM, min_samples=20, metric='haversine').fit_predict(X_rad)
#   2. Guarda {'eps_km': km, **evaluar_clustering(...)} en una lista
# Muestra la lista como DataFrame con las columnas eps_km, n_clusters, pct_ruido, silhouette
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Cuántas zonas encontró DBSCAN y qué % de entregas quedó como ruido?
- ¿En qué rango de `eps` el número de zonas se mantiene igual? ¿Qué pasa con un radio muy grande y con uno muy chico?

RELLENAR CON TUS COMENTARIOS

### 🔎 Comprobación: ¿y si hubiéramos usado euclidiana sobre grados?
Con un `eps` equivalente en grados (`eps_km / 111`, porque 1° de latitud ≈ 111 km):

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
labels_grados = DBSCAN(eps=eps_km / 111, min_samples=20).fit_predict(df[['lat', 'lon']])
print(f'Zonas con euclidiana en grados: {len(set(labels_grados) - {-1})}, ruido: {100 * (labels_grados == -1).mean():.1f}%')
print(f'Entregas con distinta asignación: {(labels_grados != labels_zona).sum()} de {len(df)}')

💡 ✍️ ¿Qué tanto cambió el resultado? ¿Por qué crees que en Monterrey la diferencia es pequeña? ¿Pasaría lo mismo en una ciudad más al norte?

RELLENAR CON TUS COMENTARIOS

---
# 5️⃣ De clusters a micro-centros (15 min)

Un cluster no es una decisión. Para cada zona necesitamos:
- **Dónde**: el **centroide** (promedio de lat y lon de sus entregas). Para zonas de pocos km, promediar grados es una buena aproximación.
- **Cuánto**: número de entregas y monto.
- **Qué tan lejos**: el **radio** que cubre al 90% de sus entregas (distancia Haversine del centroide a cada entrega). Define el área de reparto.
- **Cuándo**: la hora típica, para planear turnos.

In [ ]:
# Resumen por zona
# Pista: zonas_dbscan = resumen_zonas(df, labels_zona)
# ✍️ Tu código aquí


In [ ]:
# Mapa con los micro-centros
# Pista: plot_mapa(df, labels_zona, centros=zonas_dbscan)
# ✍️ Tu código aquí


In [ ]:
# ¿Cuántas entregas quedaron como ruido y cuánto suman?
# Pista: ruido_entregas = df[labels_zona == -1]; usa len() y ruido_entregas['monto'].sum()
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- Mira el mapa y ponle **nombre** a cada zona según su ubicación.
- ¿Qué zonas son más compactas (radio chico) y cuáles más extendidas? ¿Cuál tiene el ticket más alto? ¿Cuáles se entregan de tarde?
- ¿Cuántas entregas quedaron sueltas y cuánto suman? ¿Qué harías con ellas?

RELLENAR CON TUS COMENTARIOS

---
# 6️⃣ ¿Y si usamos K-Means? (10 min)

K-Means no acepta Haversine, así que lo ajustamos sobre la proyección en km (`X_km`). Le damos **ventaja**: le pedimos el mismo número de zonas que encontró tu DBSCAN.

In [ ]:
# 1. kmeans_entregas = KMeans(n_clusters=<zonas que encontró tu DBSCAN>, n_init=10, random_state=42).fit(X_km)
# 2. labels_kmeans = kmeans_entregas.labels_
# 3. comparacion.append(evaluar_clustering('K-Means (k=<tus zonas>, km)', X_km, labels_kmeans))
# 4. zonas_kmeans = resumen_zonas(df, labels_kmeans)
# ✍️ Tu código aquí


In [ ]:
# Mapa de K-Means con sus centros
# Pista: plot_mapa(df, labels_kmeans, centros=zonas_kmeans)
# ✍️ Tu código aquí


¿A dónde mandó K-Means las entregas sueltas? ¿Y qué tanto coinciden los dos modelos?

In [ ]:
# 1. grupo_dbscan = np.where(labels_zona == -1, 'Ruido', 'DBSCAN ' + labels_zona.astype(str))
# 2. pd.crosstab(grupo_dbscan, labels_kmeans, rownames=['DBSCAN'], colnames=['K-Means'])
# 3. con_zona = labels_zona != -1; tabla_con_zona = pd.crosstab(labels_zona[con_zona], labels_kmeans[con_zona])
#    coinciden = la suma, fila por fila, de la celda más grande. Pista: tabla_con_zona.max(axis=1).sum()
# 4. pd.DataFrame(comparacion)
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Qué tanto coinciden los dos modelos sin las entregas sueltas? ¿Y con ellas?
- ¿A qué zona de K-Means se fueron más entregas sueltas? ¿Qué le pasó a su radio del 90% comparado con DBSCAN?
- ¿Qué problema tendría un micro-centro diseñado con K-Means?

RELLENAR CON TUS COMENTARIOS

---
# 7️⃣ Conclusiones (5 min)

## ❓ Respuesta a la pregunta
> *¿Dónde abrir micro-centros de distribución y qué zona cubriría cada uno?*

✍️ Responde con tus resultados: ¿cuántas zonas encontraste y qué % de las entregas concentran? ¿En qué orden abrirías los micro-centros y por qué (entregas, radio, ticket, hora)? ¿Qué harías con las entregas sueltas?

RELLENAR CON TUS COMENTARIOS

## 📝 Lo que aprendimos
✍️ Escribe 3 cosas que aprendiste en este laboratorio.

RELLENAR CON TUS COMENTARIOS

## 🚀 Retos opcionales
- **HDBSCAN**: `HDBSCAN(min_cluster_size=50, metric='haversine', copy=True).fit(X_rad)` (de `sklearn.cluster`). ¿Cuántas zonas encuentra sin darle ningún radio? ¿Recupera más entregas de Apodaca que DBSCAN? Compara con `resumen_zonas`.
- Repite DBSCAN con `min_samples = 40` (demanda mínima más exigente). ¿Qué zonas sobreviven con el `eps` que elegiste? ¿Qué radio necesitas para recuperarlas?
- Agrega la **hora** al análisis: ¿las zonas cambian si sólo usas las entregas de la tarde (`hora >= 17`)?